In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
META_DIR = PROJECT_ROOT / "data" / "processed" / "metadata"

IAM_META_PATH        = META_DIR / "iam_metadata.csv"
EMURU_SENT_META_PATH = META_DIR / "emuru_sentences_metadata.csv"
EMURU_WORD_META_PATH = META_DIR / "emuru_words_metadata.csv"
FAIL_META_PATH       = META_DIR / "emuru_failures_metadata.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("META_DIR:", META_DIR)
print("IAM_META_PATH:", IAM_META_PATH)
print("EMURU_SENT_META_PATH:", EMURU_SENT_META_PATH)
print("EMURU_WORD_META_PATH:", EMURU_WORD_META_PATH)
print("FAIL_META_PATH:", FAIL_META_PATH)

iam_df        = pd.read_csv(IAM_META_PATH)
emuru_sent_df = pd.read_csv(EMURU_SENT_META_PATH)
emuru_word_df = pd.read_csv(EMURU_WORD_META_PATH)
fail_df       = pd.read_csv(FAIL_META_PATH)

print("\nShapes:")
print("  IAM:             ", iam_df.shape)
print("  Emuru sentences: ", emuru_sent_df.shape)
print("  Emuru words:     ", emuru_word_df.shape)
print("  Failures:        ", fail_df.shape)

print("\nIAM columns:", list(iam_df.columns))
print("Emuru sentences columns:", list(emuru_sent_df.columns))
print("Emuru words columns:", list(emuru_word_df.columns))
print("Failures columns:", list(fail_df.columns))

print("\n=== IAM head(3) ===")
display(iam_df.head(3))

print("\n=== Emuru sentences head(3) ===")
display(emuru_sent_df.head(3))

print("\n=== Emuru words head(3) ===")
display(emuru_word_df.head(3))

print("\n=== Failures head(3) ===")
display(fail_df.head(3))


In [12]:
iam_for_join = iam_df[["idx", "label", "hf_split"]].copy()

emuru_joined = emuru_sent_df.merge(
    iam_for_join,
    left_on="iam_index",  # from Emuru
    right_on="idx",       # from IAM
    how="left",
    suffixes=("", "_iam")
)

print("Joined Emuru+IAM shape:", emuru_joined.shape)
print("Joined columns (first few):", list(emuru_joined.columns[:10]), "...")


emuru_for_classifier_df = pd.DataFrame()
emuru_for_classifier_df["filepath"]      = emuru_joined["emuru_sentence_filepath"]
emuru_for_classifier_df["source"]        = "emuru"           
emuru_for_classifier_df["label"] = "fake"
emuru_for_classifier_df["text"]          = emuru_joined["iam_text"]
emuru_for_classifier_df["idx"]           = emuru_joined["iam_index"]
emuru_for_classifier_df["hf_split"]      = emuru_joined["hf_split"]
emuru_for_classifier_df["domain_label"]  = 1   # 1 = Emuru (generated)

print("\nEmuru-for-classifier shape:", emuru_for_classifier_df.shape)
print("Emuru-for-classifier columns:", list(emuru_for_classifier_df.columns))

display(emuru_for_classifier_df.head(5))


Joined Emuru+IAM shape: (9801, 10)
Joined columns (first few): ['iam_index', 'iam_filepath', 'iam_text', 'num_chunks', 'chunks_text', 'emuru_sentence_filepath', 'style_text', 'idx', 'label', 'hf_split'] ...

Emuru-for-classifier shape: (9801, 7)
Emuru-for-classifier columns: ['filepath', 'source', 'label', 'text', 'idx', 'hf_split', 'domain_label']


,filepath,source,label,text,idx,hf_split,domain_label
0,data/raw/emuru/sentences/iam_00000_sentence.png,emuru,fake,put down a resolution on the subject,0,train,1
1,data/raw/emuru/sentences/iam_00001_sentence.png,emuru,fake,and he is to be backed by Mr. Will,1,train,1
2,data/raw/emuru/sentences/iam_00002_sentence.png,emuru,fake,nominating any more Labour life Peers,2,train,1
3,data/raw/emuru/sentences/iam_00004_sentence.png,emuru,fake,"Griffiths, M P for Manchester Exchange .",4,train,1
4,data/raw/emuru/sentences/iam_00005_sentence.png,emuru,fake,is to be made at a meeting of Labour,5,train,1


In [13]:
emuru_iam_ids = emuru_sent_df["iam_index"].unique()
print("Unique IAM indices with Emuru sentences:", len(emuru_iam_ids))

iam_paired_df = iam_df[iam_df["idx"].isin(emuru_iam_ids)].copy()

print("IAM paired shape:", iam_paired_df.shape)


iam_for_classifier_df = pd.DataFrame()
iam_for_classifier_df["filepath"]     = iam_paired_df["filepath"]
iam_for_classifier_df["source"]       = "iam"       
iam_for_classifier_df["label"]        = "genuine"       
iam_for_classifier_df["text"]         = iam_paired_df["text"]
iam_for_classifier_df["idx"]          = iam_paired_df["idx"]
iam_for_classifier_df["hf_split"]     = iam_paired_df["hf_split"]
iam_for_classifier_df["domain_label"] = 0      

print("\nIAM-for-classifier shape:", iam_for_classifier_df.shape)
print("IAM-for-classifier columns:", list(iam_for_classifier_df.columns))
print("\nIAM-for-classifier label value counts:")
print(iam_for_classifier_df["label"].value_counts())

display(iam_for_classifier_df.head(5))


Unique IAM indices with Emuru sentences: 9801
IAM paired shape: (9801, 6)

IAM-for-classifier shape: (9801, 7)
IAM-for-classifier columns: ['filepath', 'source', 'label', 'text', 'idx', 'hf_split', 'domain_label']

IAM-for-classifier label value counts:
label
genuine    9801
Name: count, dtype: int64


,filepath,source,label,text,idx,hf_split,domain_label
0,data/raw/iam/iam_00000.png,iam,genuine,put down a resolution on the subject,0,train,0
1,data/raw/iam/iam_00001.png,iam,genuine,and he is to be backed by Mr. Will,1,train,0
2,data/raw/iam/iam_00002.png,iam,genuine,nominating any more Labour life Peers,2,train,0
4,data/raw/iam/iam_00004.png,iam,genuine,"Griffiths, M P for Manchester Exchange .",4,train,0
5,data/raw/iam/iam_00005.png,iam,genuine,is to be made at a meeting of Labour,5,train,0


In [14]:
# 1) Concatenate IAM and Emuru datasets
domain_clf_df = pd.concat(
    [iam_for_classifier_df, emuru_for_classifier_df],
    axis=0,
    ignore_index=True
)

# 2) Shuffle rows so classes are mixed
domain_clf_df = domain_clf_df.sample(frac=1.0, random_state=42).reset_index(drop=True)

print("Combined dataset shape:", domain_clf_df.shape)
print("Columns:", list(domain_clf_df.columns))

# 3) Label distributions
print("\nString labels (genuine vs fake):")
print(domain_clf_df["label"].value_counts())

print("\nNumeric domain_label (0=IAM, 1=Emuru):")
print(domain_clf_df["domain_label"].value_counts())

# 4) Split-wise counts (optional but useful)
print("\nCounts per split and domain_label:")
print(domain_clf_df.groupby(["hf_split", "domain_label"]).size())

print("\nHead of combined dataset:")
display(domain_clf_df.head(5))


Combined dataset shape: (19602, 7)
Columns: ['filepath', 'source', 'label', 'text', 'idx', 'hf_split', 'domain_label']

String labels (genuine vs fake):
label
genuine    9801
fake       9801
Name: count, dtype: int64

Numeric domain_label (0=IAM, 1=Emuru):
domain_label
0    9801
1    9801
Name: count, dtype: int64

Counts per split and domain_label:
hf_split    domain_label
test        0               2753
            1               2753
train       0               6128
            1               6128
validation  0                920
            1                920
dtype: int64

Head of combined dataset:


,filepath,source,label,text,idx,hf_split,domain_label
0,data/raw/iam/iam_07957.png,iam,genuine,commerce may be kept going - though if ever,7957,test,0
1,data/raw/iam/iam_09341.png,iam,genuine,"' Be silent , woman , and listen , ' Band Appa...",9341,test,0
2,data/raw/iam/iam_03294.png,iam,genuine,we would be altogether clearer in our minds,3294,train,0
3,data/raw/iam/iam_03657.png,iam,genuine,enough tacks and he got only the middle hammer...,3657,train,0
4,data/raw/emuru/sentences/iam_04302_sentence.png,emuru,fake,"purpose , the sides from one , and the bottom ...",4302,train,1


In [ ]:
DOMAIN_CLF_META_PATH = META_DIR / "domain_classification_sentences.csv"

domain_clf_df.to_csv(DOMAIN_CLF_META_PATH, index=False)

print("Saved domain classification dataset to:")
print(DOMAIN_CLF_META_PATH)
print("Shape:", domain_clf_df.shape)
print("Columns:", list(domain_clf_df.columns))


Saved domain classification dataset to:
/home/hpc/iwi5/iwi5384h/projects/handwriting_forge_project/data/processed/metadata/domain_classification_sentences.csv
Shape: (19602, 7)
Columns: ['filepath', 'source', 'label', 'text', 'idx', 'hf_split', 'domain_label']
